# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# ProbSS 6 — Choosing a classifier when time and memory matter

## What you will do

You will compare a linear classifier with a kernel classifier on the same
development split. The comparison uses balanced accuracy, fitting and
prediction time, and model size. Only after choosing a method will you use the
final test set.

Precision–recall and ROC analysis come later, with model evaluation.


In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 2026


## 1. Split the data before comparing models

The scikit-learn breast-cancer dataset is bundled with scikit-learn, so this notebook performs no download. It is a teaching benchmark, not a clinical decision tool.

Training data fit each candidate. Validation data choose between candidates. The final test data remain untouched until the choice is fixed. Stratification preserves the class proportions in these splits.


In [ ]:
dataset = load_breast_cancer()
X = dataset.data.astype(float)
y = dataset.target.astype(int)

X_development, X_test, y_development, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development,
    y_development,
    test_size=0.25,
    stratify=y_development,
    random_state=RANDOM_STATE,
)

for split_name, split_target in {
    "training": y_train,
    "validation": y_validation,
    "test": y_test,
}.items():
    assert set(np.unique(split_target)) == {0, 1}, (
        f"{split_name} split must contain both classes"
    )

print("train, validation, test:", X_train.shape, X_validation.shape, X_test.shape)
print("number of features:", X.shape[1])
print(f"dense input storage: {X.nbytes / 1e6:.3f} MB")


## 2. Compare models with one chosen score

Both candidates standardise features using parameters fitted on the training set only. Logistic regression gives an affine decision rule. The radial-basis SVC can form nonlinear boundaries but stores support vectors and can cost more at prediction time.

For binary labels, let $N_0$ and $N_1$ be the numbers of validation observations in classes 0 and 1. When both are positive, define

$$
\operatorname{balanced\ accuracy}
=\frac12\left(
\frac{\#\{i:y_i=0,\widehat y_i=0\}}{N_0}
+\frac{\#\{i:y_i=1,\widehat y_i=1\}}{N_1}
\right).
$$

It is the mean of the two class-wise correct-classification rates, so each class receives equal weight even when the class counts differ. The stratified splits above contain both classes; the assertions make the positive-denominator requirement explicit. This practical definition is supplied here because the formal metrics treatment comes in Lecture 12.


In [ ]:
candidates = {
    "linear logistic": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=4_000, random_state=RANDOM_STATE),
    ),
    "RBF kernel SVC": make_pipeline(
        StandardScaler(),
        SVC(C=2.0, kernel="rbf"),
    ),
}

fitted_candidates = {}
rows = []
for name, procedure in candidates.items():
    start = perf_counter()
    fitted = clone(procedure).fit(X_train, y_train)
    fit_seconds = perf_counter() - start

    start = perf_counter()
    prediction = fitted.predict(X_validation)
    predict_seconds = perf_counter() - start

    model = fitted.steps[-1][1]
    if hasattr(model, "support_vectors_"):
        parameter_bytes = (
            model.support_vectors_.nbytes
            + model.dual_coef_.nbytes
            + model.intercept_.nbytes
        )
    else:
        parameter_bytes = model.coef_.nbytes + model.intercept_.nbytes

    rows.append(
        {
            "procedure": name,
            "validation accuracy": accuracy_score(y_validation, prediction),
            "validation balanced accuracy": balanced_accuracy_score(
                y_validation, prediction
            ),
            "fit milliseconds": 1_000 * fit_seconds,
            "predict milliseconds": 1_000 * predict_seconds,
            "stored model MB": parameter_bytes / 1e6,
        }
    )
    fitted_candidates[name] = fitted

comparison = pd.DataFrame(rows).set_index("procedure")
comparison


Wall-clock timings vary by machine and load. They are measurements, not mathematical constants. The storage estimate counts the main learned arrays, not the Python object overhead.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, row in comparison.iterrows():
    ax.scatter(
        row["fit milliseconds"],
        row["validation balanced accuracy"],
        s=80,
        label=name,
    )
ax.set(
    xlabel="fit time (milliseconds on this run)",
    ylabel="validation balanced accuracy",
    title="Predictive and computational validation criteria",
)
ax.legend()
ax.grid(alpha=0.2)
plt.show()


## 3. Choose the model, then use the test set once

The automatic rule below selects the highest validation balanced accuracy and breaks an exact tie in favour of the smaller stored model. In a deployment with a strict latency or memory budget, that budget should be stated before selection and encoded in the rule.


In [ ]:
choice = max(
    comparison.index,
    key=lambda name: (
        comparison.loc[name, "validation balanced accuracy"],
        -comparison.loc[name, "stored model MB"],
    ),
)
final_procedure = clone(candidates[choice]).fit(X_development, y_development)
test_prediction = final_procedure.predict(X_test)

print("Selected procedure:", choice)
print(f"Final test accuracy: {accuracy_score(y_test, test_prediction):.3f}")
print(
    "Final test balanced accuracy:",
    f"{balanced_accuracy_score(y_test, test_prediction):.3f}",
)


In [ ]:
resource_summary = pd.DataFrame(
    {
        "question": [
            "How many input features are processed per observation?",
            "How much dense feature memory does the full table use?",
            "What does the kernel model retain?",
            "What is not measured here?",
        ],
        "answer": [
            X.shape[1],
            f"{X.nbytes / 1e6:.3f} MB",
            "support vectors and coefficients",
            "energy, concurrency, preprocessing overhead, and clinical cost",
        ],
    }
)
resource_summary


## Recap

Before you finish, make sure you can:

1. Explain why you chose the classifier, using predictive performance and at least one measure of time or storage.
2. Give one practical limit that could make you choose the other classifier.
3. Explain why looking at the final test result before choosing would make that result unreliable.
4. Record the hardware/software context whenever reporting timings.
